# Exercice 2 : Pseudonymiser les IP et les horodatages sans casser la détection

> **Objectif.** Rendre les IP et les dates de UNSW-NB15 non lisibles par un tiers (CryptoPAn + décalage temporel secret),
> puis vérifier que la **baseline comportementale** de chaque machine, et donc la détection d'anomalies, est identique.

**Volets du plan de cours (séance 12) :** *préservation de la confidentialité des données* et *techniques d'anonymisation* (ici la pseudonymisation, et pourquoi elle n'est pas une anonymisation).

**Lien avec l'intra (partie 3, Q6).** La question demandait s'il fallait retirer les IP sources et les horodatages avant d'entraîner un modèle de détection. Clé : c'était un piège, ces colonnes sont **nécessaires** pour établir une baseline comportementale.
Cet exercice montre la suite logique : on les garde, mais **pseudonymisées**.

**Dépendances :** `pandas`, `scikit-learn`, `cryptography` (AES pour CryptoPAn), tous installés par l'étape 0.

**Les données.** `data/unsw_nb15_flux_bruts_60k.csv.gz` : 60 000 flux tirés au hasard (graine 0) parmi les 2 059 415 flux « bruts » de **UNSW-NB15** (Moustafa et Slay, MilCIS 2015), **avec** IP, ports et horodatages (`stime`), contrairement aux fichiers *training/testing-set* des chapitres précédents.
Fichier source : `Network-Flows/UNSW_Flow.parquet` du dépôt Hugging Face [rdpahalavan/UNSW-NB15](https://huggingface.co/datasets/rdpahalavan/UNSW-NB15). Licence du jeu original : « libre pour la recherche universitaire », citation obligatoire ([research.unsw.edu.au/projects/unsw-nb15-dataset](https://research.unsw.edu.au/projects/unsw-nb15-dataset)).

**Le plan.**
1. Regarder les IP et les horodatages bruts.
2. Deux fausses bonnes idées : supprimer, hacher.
3. CryptoPAn : un pseudonyme qui conserve les sous-réseaux.
4. Un décalage temporel secret et cohérent.
5. Preuve : la baseline et le détecteur d'anomalies donnent les mêmes résultats.
6. Limites : pseudonymisé n'est pas anonyme.

## Étape 0 : exécutez cette cellule en premier
Elle installe les paquets manquants et, sur Colab, récupère `helper.py` et `data/` depuis GitHub. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances de ce notebook : scikit-learn cryptography
import os, sys, importlib.util
if not os.path.exists('helper.py'):   # notebook ouvert depuis Google Drive / Colab
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 7 - IA et vie privée"
manquants = [pip for module, pip in {'sklearn': 'scikit-learn', 'cryptography': 'cryptography'}.items() if importlib.util.find_spec(module) is None]
if manquants:
    %pip install -q {' '.join(manquants)}
print('Prêt :', os.path.exists('helper.py') and os.path.exists('data'))

Prêt : True


## Étape 1 : Les flux bruts
**Ce qu'on fait :** on charge les flux et on regarde qui parle à qui, et quand.

In [2]:
from helper import charger_flux
flux = charger_flux()
flux[['debut', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'service', 'sbytes', 'attack_label']].head()

,debut,source_ip,source_port,destination_ip,destination_port,service,sbytes,attack_label
0,2015-01-22 11:50:17,59.166.0.0,21702,149.171.126.0,61361,NaN,4654,normal
1,2015-01-22 11:50:27,59.166.0.0,28852,149.171.126.3,53,dns,132,normal
2,2015-01-22 11:50:29,59.166.0.2,56170,149.171.126.8,54430,NaN,3806,normal
3,2015-01-22 11:50:29,59.166.0.6,45559,149.171.126.2,80,http,1052,normal
4,2015-01-22 11:50:30,59.166.0.8,17542,149.171.126.4,53,dns,132,normal


In [3]:
# Les sous-réseaux /24 (les 3 premiers octets) des IP sources
flux['source_ip'].str.rsplit('.', n=1).str[0].value_counts()

source_ip
59.166.0       54971
175.45.176      4059
149.171.126      655
10.40.182        137
10.40.85         136
10.40.170         39
192.168.241        3
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** le banc d'essai de l'UNSW a trois réseaux : `59.166.0.x` (utilisateurs normaux), `175.45.176.x` (machines d'attaque) et `149.171.126.x` (serveurs).
La **structure en sous-réseaux** est une information précieuse pour l'analyste : « tout le /24 175.45.176 attaque ».

## Étape 2 : Deux fausses bonnes idées
**Idée 1 : supprimer les colonnes.** Plus d'IP = plus de profil par machine, plus de « cette IP n'a jamais parlé à ce serveur ». C'est la mauvaise réponse à la Q6 de l'intra.

**Idée 2 : hacher les IP (SHA-256).** Le résultat semble illisible… mais l'espace IPv4 ne compte que 2³² ≈ 4,3 milliards d'adresses. Un attaquant hache **toutes** les adresses candidates et inverse la table.

In [4]:
import hashlib, ipaddress, time

def hacher(ip):
    return hashlib.sha256(ip.encode()).hexdigest()[:12]

publie = hacher('175.45.176.3')                       # ce que contiendrait le fichier « anonymisé »
print('Pseudonyme publié :', publie)

debut = time.time()
table = {hacher(str(ip)): str(ip) for ip in ipaddress.ip_network('175.45.0.0/16')}   # 65 536 adresses
print('Retrouvé :', table[publie], f'en {time.time() - debut:.2f} s')

Pseudonyme publié : 31f230da98b9


Retrouvé : 175.45.176.3 en 0.33 s


**👀 Ce qu'on doit voir :** l'adresse est retrouvée en moins d'une seconde. Les 4,3 milliards d'adresses IPv4, c'est 65 536 fois plus : quelques heures sur un portable, quelques minutes sur un GPU. **Un hachage sans clé secrète n'est pas une pseudonymisation.**
Et en plus, le hachage détruit les sous-réseaux : `175.45.176.1` et `175.45.176.3` reçoivent des pseudonymes sans aucun lien.

## Étape 3 : CryptoPAn, la pseudonymisation qui conserve les préfixes
**Ce qu'on fait :** CryptoPAn (Xu, Fan, Ammar et Moon, ICNP 2002) chiffre l'adresse **bit par bit** avec AES et une **clé secrète** de 32 octets.
Le bit *i* du pseudonyme ne dépend que des *i* premiers bits de l'adresse : deux IP qui partagent un préfixe de *k* bits ont des pseudonymes qui partagent aussi *k* bits.
Notre implémentation (dans `helper.py`, 15 lignes) a été vérifiée contre l'implémentation de référence `yacryptopan`.

In [5]:
import os
from helper import CryptoPAn

cle = hashlib.sha256(b'cle de demo 8IAR805').digest()   # en production : os.urandom(32), gardée dans un coffre (HSM, Vault)
cpan = CryptoPAn(cle)

for ip in ['59.166.0.1', '59.166.0.2', '175.45.176.1', '175.45.176.3', '149.171.126.6']:
    print(f'{ip:>15}  ->  {cpan.anonymiser(ip)}')

     59.166.0.1  ->  36.86.63.255
     59.166.0.2  ->  36.86.63.253
   175.45.176.1  ->  144.211.134.6
   175.45.176.3  ->  144.211.134.4
  149.171.126.6  ->  171.164.125.197


**👀 Ce qu'on doit voir :** `59.166.0.1` et `59.166.0.2` reçoivent des pseudonymes qui partagent leurs 3 premiers octets ; même chose pour les deux `175.45.176.x`. Les sous-réseaux survivent, mais sous un autre nom.

In [6]:
pseudo = flux.copy()
pseudo['source_ip'] = flux['source_ip'].map(cpan.anonymiser)
pseudo['destination_ip'] = flux['destination_ip'].map(cpan.anonymiser)
pseudo['source_ip'].str.rsplit('.', n=1).str[0].value_counts()

source_ip
36.86.63       54971
144.211.134     4059
171.164.125      655
10.214.141       137
10.214.91        136
10.214.149        39
201.81.15          3
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** exactement les mêmes effectifs par /24 qu'à l'étape 1, avec des préfixes différents.

## Étape 4 : Décaler le temps
**Ce qu'on fait :** une date absolue permet de relier les flux à des événements publics (« l'attaque du 18 février 2015 »). On décale **tous** les horodatages du même nombre secret de semaines.
- Un **seul** décalage pour tout le jeu : les intervalles entre flux sont conservés exactement.
- Un nombre **entier de semaines** : l'heure de la journée et le jour de la semaine sont conservés (« connexion à 3 h du matin un dimanche » reste visible).

In [7]:
import pandas as pd

decalage = pd.Timedelta(weeks=int.from_bytes(hashlib.sha256(cle + b'temps').digest()[:2], 'big') % 520 + 52)   # 1 à 11 ans, dérivé de la clé
pseudo['debut'] = flux['debut'] + decalage
pseudo = pseudo.drop(columns=['stime', 'ltime'])

print('Original    :', flux['debut'].min(), '->', flux['debut'].max())
print('Pseudonymisé:', pseudo['debut'].min(), '->', pseudo['debut'].max())
print('Même jour de semaine et même heure partout :',
      (flux['debut'].dt.day_of_week == pseudo['debut'].dt.day_of_week).all() and (flux['debut'].dt.hour == pseudo['debut'].dt.hour).all())

Original    : 2015-01-22 11:50:17 -> 2015-02-18 12:21:08
Pseudonymisé: 2025-04-17 11:50:17 -> 2025-05-14 12:21:08
Même jour de semaine et même heure partout : True


## Étape 5 : La baseline est-elle conservée ?
**Ce qu'on fait :** on calcule le profil de chaque IP source (nombre de flux, volume, destinations, ports, heure d'activité, rythme) sur les deux versions, puis on les compare ligne à ligne via la table de correspondance (que seul le détenteur de la clé peut construire).

In [8]:
from helper import baseline_par_ip

base_orig = baseline_par_ip(flux)
base_pseudo = baseline_par_ip(pseudo)
base_orig.head(6)

,nb_flux,octets_moyens,destinations,ports_dst,heure_mediane,intervalle_median_s,part_attaques
source_ip,,,,,,,
59.166.0.2,5665,4521.0,10,2268,12.0,10.0,0.0
59.166.0.5,5648,4444.0,10,2192,12.0,10.0,0.0
59.166.0.0,5601,4576.0,10,2220,12.0,11.0,0.0
59.166.0.4,5592,4536.0,10,2200,12.0,10.0,0.0
59.166.0.3,5559,4574.0,10,2240,12.0,10.0,0.0
59.166.0.1,5485,4593.0,10,2142,12.0,11.0,0.0


In [9]:
correspondance = {ip: cpan.anonymiser(ip) for ip in base_orig.index}
identiques = base_orig.rename(index=correspondance).sort_index().equals(base_pseudo.sort_index())
print('Profils identiques pour les', len(base_orig), 'IP sources :', identiques)

Profils identiques pour les 40 IP sources : True


**Ce qu'on fait :** dernière preuve, on entraîne un détecteur d'anomalies (Isolation Forest, vu au Ch. 2) sur les profils, une fois sur chaque version.

In [10]:
from sklearn.ensemble import IsolationForest

def ip_suspectes(profils):
    X = profils.drop(columns='part_attaques').sort_index()
    modele = IsolationForest(contamination=0.15, random_state=0).fit(X)
    return set(X.index[modele.predict(X) == -1])

suspectes_orig = ip_suspectes(base_orig.rename(index=correspondance))   # mêmes noms de lignes des deux côtés
suspectes_pseudo = ip_suspectes(base_pseudo)
inverse = {v: k for k, v in correspondance.items()}
print('Mêmes IP signalées :', suspectes_orig == suspectes_pseudo)
print('IP signalées, ré-identifiées par le détenteur de la clé :', sorted(inverse[ip] for ip in suspectes_pseudo))

Mêmes IP signalées : True
IP signalées, ré-identifiées par le détenteur de la clé : ['149.171.126.2', '149.171.126.3', '149.171.126.6', '149.171.126.7', '175.45.176.1', '175.45.176.2']


**👀 Ce qu'on doit voir :** `True` deux fois, et parmi les IP signalées, deux machines d'attaque `175.45.176.x`. Le SOC (ou le prestataire externe, ou l'équipe de science des données) travaille sur les pseudonymes et obtient **exactement** les mêmes alertes ; seul le détenteur de la clé peut ré-identifier une IP signalée pour agir.

## Étape 6 : Pseudonymisé n'est pas anonyme
**Ce qu'on fait :** un attaquant obtient le fichier pseudonymisé. Il connaît, par observation externe, les serveurs les plus sollicités du réseau (c'est souvent public : le site web, le DNS). Il classe les destinations du fichier par nombre de flux et aligne les rangs.

In [ ]:
vraies = flux['destination_ip'].value_counts().head(3)        # ce que l'attaquant sait déjà
pseudos = pseudo['destination_ip'].value_counts().head(3)     # ce qu'il lit dans le fichier
pd.DataFrame({'IP connue de l attaquant': vraies.index, 'pseudonyme au même rang': pseudos.index,
              'bonne correspondance ?': [cpan.anonymiser(ip) == p for ip, p in zip(vraies.index, pseudos.index)]})

,IP connue de l attaquant,pseudonyme au même rang,bonne correspondance ?
0,149.171.126.0,171.164.125.192,True
1,149.171.126.1,171.164.125.193,True
2,149.171.126.4,171.164.125.199,True


**👀 Ce qu'on doit voir :** trois correspondances justes sur trois. Le comportement (volume, rythme, ports) est une **empreinte**. Une seule IP reconnue par son comportement, et CryptoPAn révèle aussi une partie du préfixe de ses voisines : c'est la limite connue des schémas qui conservent les préfixes, analysée dans l'article de Xu *et al.* lui-même.

Conséquence juridique : sous la Loi 25, un renseignement **dépersonnalisé** (il ne permet plus d'identifier *directement* la personne) reste un renseignement personnel ; seul un renseignement **anonymisé**, de façon irréversible, sort de la loi. Le RGPD dit la même chose des données pseudonymisées (considérant 26). La pseudonymisation réduit le risque et la gravité d'une fuite, elle ne dispense pas des obligations.

## Ce qu'il faut retenir
- Supprimer les IP et les horodatages détruit la baseline (réponse à la Q6 de l'intra). Les **hacher sans clé** ne protège rien : l'espace IPv4 s'énumère.
- **CryptoPAn** (clé secrète + AES) donne des pseudonymes cohérents qui conservent les sous-réseaux ; un **décalage secret d'un nombre entier de semaines** conserve les rythmes horaires et hebdomadaires.
- Résultat mesuré : profils par IP et alertes de l'Isolation Forest **identiques** avant et après.
- Pseudonymiser n'est pas anonymiser : le comportement reste une empreinte, la clé doit être protégée, et la donnée reste personnelle au sens de la loi.

**Exercice suivant :** le même problème côté santé. Que vaut un fichier « sans nom » quand il reste le code postal, la date de naissance et le sexe ?